In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from skimpy import clean_columns
import requests
import re

## LA County 2020 Census Tracts Data

In [2]:
# read in LA county census tracts data
cts = gpd.read_file('../data/LA_County_2020_Census_Tracts.geojson')

In [3]:
# preview
cts.head()

,OBJECTID,CT20,LABEL,ShapeSTArea,ShapeSTLength,geometry
0,4992,101110,1011.10,1.229562e+07,15083.854287,"POLYGON ((-118.29793 34.26323, -118.30082 34.2..."
1,4993,101122,1011.22,2.845774e+07,31671.455844,"POLYGON ((-118.27743 34.25991, -118.27743 34.2..."
2,4994,101220,1012.20,7.522093e+06,12698.783810,"POLYGON ((-118.27818 34.25577, -118.27887 34.2..."
3,4995,101221,1012.21,3.812000e+06,9161.710543,"POLYGON ((-118.28735 34.25591, -118.28863 34.2..."
4,4996,101222,1012.22,3.191371e+06,9980.600461,"POLYGON ((-118.28594 34.2559, -118.28697 34.25..."


In [4]:
# check crs
cts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [5]:
# drop unnecessary columns
cts = cts.drop(columns = [
    'OBJECTID',
    'CT20',
    'ShapeSTLength',
    'ShapeSTArea'
])

In [6]:
# rename some columns
cts = cts.rename(columns = {
    'LABEL': 'TRACT'
})

In [7]:
# set TRACT values as float
cts['TRACT'] = cts['TRACT'].astype(float)

In [8]:
# preview dataset
cts.head()

,TRACT,geometry
0,1011.10,"POLYGON ((-118.29793 34.26323, -118.30082 34.2..."
1,1011.22,"POLYGON ((-118.27743 34.25991, -118.27743 34.2..."
2,1012.20,"POLYGON ((-118.27818 34.25577, -118.27887 34.2..."
3,1012.21,"POLYGON ((-118.28735 34.25591, -118.28863 34.2..."
4,1012.22,"POLYGON ((-118.28594 34.2559, -118.28697 34.25..."


## Calfornia SVI Data

In [9]:
# read in CA SVI data
ca_svi = pd.read_csv('../data/California_2020_SVI.csv')

In [10]:
# preview data
ca_svi.head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,6,California,CA,6001,Alameda,6001400100,"Census Tract 4001, Alameda County, California",2.681809,3035,402,...,14.0,3.0,0.0,1.3,0.0,1.3,5.6,3.9,0.6,0.9
1,6,California,CA,6001,Alameda,6001400200,"Census Tract 4002, Alameda County, California",0.226472,1983,209,...,11.0,4.7,0.3,0.4,0.0,2.0,10.6,5.0,0.4,0.6
2,6,California,CA,6001,Alameda,6001400300,"Census Tract 4003, Alameda County, California",0.428898,5058,559,...,15.3,7.6,0.1,0.3,0.7,1.1,4.1,3.7,1.1,1.4
3,6,California,CA,6001,Alameda,6001400400,"Census Tract 4004, Alameda County, California",0.276502,4179,529,...,10.0,3.3,0.6,0.8,0.0,1.0,6.7,2.7,0.1,0.2
4,6,California,CA,6001,Alameda,6001400500,"Census Tract 4005, Alameda County, California",0.228350,4021,631,...,9.6,3.9,0.0,1.0,0.0,1.0,9.4,5.2,0.0,1.0


In [11]:
# only keep observations within LA county
svi = ca_svi[ca_svi['COUNTY'] == 'Los Angeles']

In [12]:
# reset index & drop old index
svi = ca_svi.reset_index(drop = True)

In [13]:
# drop unnecessary columns
svi = svi.drop(columns = [
    'ST',
    'STATE',
    'ST_ABBR',
    'STCNTY',
    'COUNTY',
    'FIPS',
    'AREA_SQMI'
])

In [14]:
# retrieve census tract number
svi['LOCATION'] = svi['LOCATION'].astype(str)
svi['TRACT'] = svi['LOCATION'].str.extract(r'(\d+\.\d+|\d+)')
svi['TRACT'] = svi['TRACT'].astype(float)

/var/folders/cn/cp_jck311xn7c351b70mtr1h0000gn/T/ipykernel_43587/456375727.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  svi['TRACT'] = svi['LOCATION'].str.extract(r'(\d+\.\d+|\d+)')


In [15]:
# drop location (no longer needed)
svi = svi.drop(columns = ['LOCATION'])

In [16]:
# preview cleaned data
svi.head()

,E_TOTPOP,M_TOTPOP,E_HU,M_HU,E_HH,M_HH,E_POV150,M_POV150,E_UNEMP,M_UNEMP,...,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE,TRACT
0,3035,402,1411,134,1274,125,205,99,16,21,...,3.0,0.0,1.3,0.0,1.3,5.6,3.9,0.6,0.9,4001.0
1,1983,209,856,83,830,79,138,39,98,77,...,4.7,0.3,0.4,0.0,2.0,10.6,5.0,0.4,0.6,4002.0
2,5058,559,2674,227,2419,245,430,182,111,70,...,7.6,0.1,0.3,0.7,1.1,4.1,3.7,1.1,1.4,4003.0
3,4179,529,1884,140,1740,144,498,228,66,69,...,3.3,0.6,0.8,0.0,1.0,6.7,2.7,0.1,0.2,4004.0
4,4021,631,1767,297,1643,295,513,244,169,93,...,3.9,0.0,1.0,0.0,1.0,9.4,5.2,0.0,1.0,4005.0


## Merging Census Tract Data & SVI Data

In [17]:
# merge dataframes on tract number
svi_tracts = cts.merge(svi, how = 'outer', on = 'TRACT')

In [18]:
# replace -999 values with NaN
svi_tracts = svi_tracts.replace(-999, np.nan)

In [19]:
# column names to snake case
svi_tracts = clean_columns(svi_tracts)

In [20]:
# preview cleaned data
svi_tracts.head()

,tract,geometry,e_totpop,m_totpop,e_hu,m_hu,e_hh,m_hh,e_pov_150,m_pov_150,...,ep_asian,mp_asian,ep_aian,mp_aian,ep_nhpi,mp_nhpi,ep_twomore,mp_twomore,ep_otherrace,mp_otherrace
0,1.0,None,5537.0,406.0,1850.0,203.0,1715.0,209.0,1161.0,395.0,...,0.0,0.7,0.2,0.3,0.0,0.7,4.9,1.9,0.0,0.7
1,1.0,None,3852.0,496.0,854.0,153.0,804.0,159.0,592.0,159.0,...,4.3,2.4,1.0,0.8,0.0,1.0,2.0,1.2,1.4,1.6
2,1.0,None,4337.0,750.0,2287.0,278.0,1957.0,284.0,2101.0,522.0,...,8.2,4.5,3.5,2.2,0.0,0.9,7.1,4.3,1.0,1.5
3,1.0,None,2832.0,328.0,1248.0,126.0,1162.0,133.0,502.0,240.0,...,0.8,0.8,1.6,2.0,0.0,1.4,2.5,2.5,0.6,1.0
4,1.0,None,3225.0,400.0,1236.0,154.0,1091.0,149.0,422.0,154.0,...,4.6,5.1,0.5,0.7,0.2,0.4,0.8,0.6,0.4,0.5


In [21]:
# drop census tracts not in continental LA county
svi_tracts_continental = svi_tracts.drop([2059, 2060])

In [22]:
# save dataset
svi_tracts_continental.to_file('../data/svi_tracts.geojson')